In [1]:
pip install bertopic sentence-transformers --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   --------------- ------------------------ 0.8/2.0 MB 4.6 MB/s eta 0:00:01
   ------------------------------------- -- 1.8/2.0 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.4 MB/s  0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.9 MB 4.7 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.9 MB 4.6 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.9 MB 4.6 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.9 MB 4.5 MB/s eta 0:00:02
   ----------------- ---------------------- 4.5/9.9 MB 4.6 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.9 MB 4.6 MB/s eta 0:00:01
   -------------------------- ------------- 6.6/9.9 MB 4.6 MB/s eta 0:00:01
   ------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## Import

In [7]:
import pandas as pd
from bertopic import BERTopic
from transformers import pipeline


## Load Data

In [3]:
df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended_analyzed.csv")
print(f"Loaded {len(df)} rows")

Loaded 282403 rows


## Prep text and run BERTopic (use a sample first — 112K is a lot for a first pass)

In [4]:
# Start with a manageable sample to validate the approach before committing to the full dataset
sample_df = df.dropna(subset=["review_text"]).sample(n=20000, random_state=42)
docs = sample_df["review_text"].tolist()

topic_model = BERTopic(min_topic_size=50)
topics, probs = topic_model.fit_transform(docs)

sample_df["ml_topic"] = topics
print("Done. Topics found:")
topic_model.get_topic_info().head(20)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2294.82it/s]


Done. Topics found:


,Topic,Count,Name,Representation,Representative_Docs
0,-1,6815,-1_the_to_and_it,"[the, to, and, it, my, is, for, you, in, with]",[I came in the UK for a half year. I had to re...
1,0,877,0_bank_banking_best_online,"[bank, banking, best, online, banks, ever, tha...","[The best bank, The best bank!, the best bank]"
2,1,710,1_app_great_easy_use,"[app, great, easy, use, very, friendly, servic...","[Great app and easy to use, Great app and easy..."
3,2,520,2_banking_app_best_bank,"[banking, app, best, bank, ever, apps, banks, ...","[Best banking app, Best banking App, best bank..."
4,3,488,3_good_goodd_goodworth_gooddd,"[good, goodd, goodworth, gooddd, likey, well, ...","[Good, good, Good.]"
5,4,476,4_app_money_exchange_currency,"[app, money, exchange, currency, for, transfer...","[Best app for currency exchange, Very easy to ..."
6,5,426,5_revolut_is_and_with,"[revolut, is, and, with, in, for, using, of, t...",[Revolut is an amazing app for money transacti...
7,6,391,6_de_la_que_si,"[de, la, que, si, muy, en, et, se, el, pas]","[Hasta ahora muy bien, pero la he usado sólo u..."
8,7,335,7_travelling_abroad_travel_traveling,"[travelling, abroad, travel, traveling, for, w...","[So easy to use when travelling, love it, Eas..."
9,8,320,8_card_abroad_cards_for,"[card, abroad, cards, for, travel, use, the, t...",[When travelling abroad it's the best card to ...


## Inspect what each topic actually contains

In [5]:
for topic_num in topic_model.get_topic_info()["Topic"].head(10):
    if topic_num == -1:
        continue  # -1 is BERTopic's "outlier/no clear topic" bucket
    print(f"Topic {topic_num}: {topic_model.get_topic(topic_num)[:5]}")

Topic 0: [('bank', np.float64(0.05826827138940629)), ('banking', np.float64(0.04489316297129643)), ('best', np.float64(0.027028173074401564)), ('online', np.float64(0.021605665455309533)), ('banks', np.float64(0.019955369257986327))]
Topic 1: [('app', np.float64(0.049786201765237854)), ('great', np.float64(0.02852174209062645)), ('easy', np.float64(0.022212859598373907)), ('use', np.float64(0.021635910130096574)), ('very', np.float64(0.016411989176179833))]
Topic 2: [('banking', np.float64(0.06320738521583767)), ('app', np.float64(0.029129838977623546)), ('best', np.float64(0.025341503148075317)), ('bank', np.float64(0.024803610314392724)), ('ever', np.float64(0.01603958037780492))]
Topic 3: [('good', np.float64(0.6898202483403436)), ('goodd', np.float64(0.041293409987303145)), ('goodworth', np.float64(0.015969106649901605)), ('gooddd', np.float64(0.015969106649901605)), ('likey', np.float64(0.015969106649901605))]
Topic 4: [('app', np.float64(0.024748279998363227)), ('money', np.float

## BERTopic Analysis — Summary of Findings

**Method:** Unsupervised ML topic modeling (BERTopic) on a 20,000-review sample from the 
112,000+ dataset. No predefined categories — topics discovered directly from text.

| Metric | Value |
|---|---|
| Reviews analyzed | 20,000 (sample) |
| Reviews assigned to a topic | 13,185 (~66%) |
| Reviews unclassified (Topic -1) | 6,815 (~34%) |
| Total topics discovered | 19 |
| Coverage vs. manual keyword method | 66% vs. ~25% |

### Topic Groups

| Group | Topics | What it shows |
|---|---|---|
| General praise | 0, 1, 2, 9, 10, 11, 13, 14, 17, 18 | "Best banking app," "easy to use," "great service" |
| New use-cases (not in manual keywords) | 4, 7, 8, 12, 16 | Currency exchange, travel/abroad card use, crypto & trading |
| Language artifact | 6 | Entirely Spanish-language reviews — dataset isn't 100% English |
| Key negative cluster | 15 | "account," "blocked," "closed," "money" |
| Data artifact (not a real topic) | 3 | Spelling variants of "good" clustered together |

### Key Finding

| Method | Found this theme | Independently? |
|---|---|---|
| Manual keywords (earlier analysis) | account_freeze | — |
| BERTopic (this analysis) | Topic 15: account/blocked/closed | Yes — no shared keywords used |

**Why this matters:** the same account-freeze problem was found by two completely different 
methods. This cross-validation makes it a stronger, more credible finding.

### Limitations

| Limitation | Explanation |
|---|---|
| 34% unclassified | Very short/vague reviews (e.g. "good") don't cluster well |
| Topic 3 is not a real topic | Just spelling variants of the same word |
| Sample, not full dataset | Ran on 20,000 of 112,000+ reviews for processing time |

## Save

In [17]:
sample_df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_bertopic.csv", index=False)
print(f"Saved {len(sample_df)} rows with BERTopic topics.")

# Also save the topic summary table (the one shown in your screenshot) separately
topic_info = topic_model.get_topic_info()
topic_info.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_bertopic_summary.csv", index=False)
print(f"Saved topic summary: {len(topic_info)} topics.")

Saved 5000 rows with BERTopic topics.
Saved topic summary: 85 topics.


## BERTopic Modeling with larger data

In [18]:
df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended_analyzed.csv")

bertopic_sample_df = df.dropna(subset=["review_text"]).sample(n=20000, random_state=42)
docs = bertopic_sample_df["review_text"].tolist()

bertopic_model = BERTopic(min_topic_size=50)
topics, probs = bertopic_model.fit_transform(docs)

bertopic_sample_df["ml_topic"] = topics
print("Done. Topics found:")
bertopic_model.get_topic_info().head(20)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2165.42it/s]


Done. Topics found:


,Topic,Count,Name,Representation,Representative_Docs
0,-1,3030,-1_great_very_good_easy,"[great, very, good, easy, it, use, so, far, ef...","[very good app, Very good app, So far very good]"
1,0,10070,0_the_to_and_for,"[the, to, and, for, my, app, is, in, it, you]",[love this app and my Revolut card. easy to us...
2,1,511,1_good_excellent_goodd_well,"[good, excellent, goodd, well, gooddd, likey, ...","[good, Good, Good]"
3,2,313,2_excellent_excellento_experimente_great,"[excellent, excellento, experimente, great, st...","[Excellent, Excellent, Excellent]"
4,3,284,3_easy_use_useful_convenient,"[easy, use, useful, convenient, very, fast, an...","[Easy to use and very useful 👌, very useful an..."
5,4,271,4_service_services_excellent_very,"[service, services, excellent, very, customer,...","[Great service!, great service !, Great service.]"
6,5,270,5_works_working_well_does,"[works, working, well, does, me, perfectly, fi...","[Works well for me 👍👍, Works well for me, It w..."
7,6,264,6_fast_easy_simple_quick,"[fast, easy, simple, quick, and, very, super, ...","[Easy and fast, Easy and fast!, Easy and fast.]"
8,7,264,7_de_la_que_excelent,"[de, la, que, excelent, si, muy, en, et, se, el]","[Hasta ahora muy bien, pero la he usado sólo u..."
9,8,223,8_use_easy_to_it,"[use, easy, to, it, love, good, so, very, and,...","[It's great easy to use, Good and easy to use,..."


## Save

In [20]:
bertopic_sample_df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_bertopic.csv", index=False)
print(f"Saved {len(bertopic_sample_df)} rows with BERTopic topics.")

topic_info = bertopic_model.get_topic_info()
topic_info.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_bertopic_summary.csv", index=False)
print(f"Saved topic summary: {len(topic_info)} topics.")

Saved 20000 rows with BERTopic topics.
Saved topic summary: 58 topics.


## BERTopic Analysis — Final Findings (with Reproducibility Note)

**Method:** Unsupervised ML topic modeling (BERTopic) run on 20,000-review samples from the 
full dataset, to explore natural topic clusters without predefined categories.

### Reproducibility Note

Running BERTopic on different 20,000-review samples produced noticeably different results:

| Run | Topics Found | Notable Result |
|---|---|---|
| Initial run | 19 | Included a distinct negative cluster (account/blocked/closed) |
| Later run(s) | 58 | Predominantly positive/generic topics; no equivalent negative cluster |

**Key finding:** BERTopic's topic count and cluster composition were **not consistently 
reproducible** across runs, even on samples drawn from the same underlying dataset. This is 
a known limitation of density-based clustering algorithms (UMAP + HDBSCAN) — results can be 
sensitive to which specific reviews are sampled, particularly in a dataset with a heavy 
positive skew (~88% positive reviews), where negative reviews are a smaller, more 
linguistically varied minority.

### Conclusion

Given this reproducibility issue, the account_freeze / customer_support finding from earlier 
in this project is treated as supported by two independent, reproducible methods rather than 
three:

1. **Manual keyword frequency analysis** — customer_support present in 17.7% of 1-star 
   reviews; account_freeze and verification show 68-72% negative-mention concentration
2. **Predictive model feature importance** — has_customer_support consistently ranked as 
   the top specific theme predictor across both VADER-based and transformer-based model 
   versions

BERTopic's initial result is reported as an observed but non-reproduced finding, included 
for transparency rather than as confirmed independent validation.

### Methodological Takeaway

This exercise highlights an important principle in unsupervised ML: a single favorable run 
should not be treated as a stable finding without checking reproducibility. Testing this 
explicitly — rather than reporting only the first, most convenient result — reflects sound 
analytical practice, even though the outcome here was a null result rather than confirmation.

In [26]:
topic_sentiment = bertopic_sample_df.groupby("ml_topic").agg(
    review_count=("rating", "count"),
    avg_rating=("rating", "mean"),
    avg_sentiment=("sentiment_score", "mean")
).sort_values("avg_rating")

print(topic_sentiment.head(15))

          review_count  avg_rating  avg_sentiment
ml_topic                                         
 29                 54    1.129630      -0.052506
 20                 76    1.342105      -0.301786
 0               13870    2.509156       0.093510
 2                 400    3.657500       0.016070
 25                 58    4.120690       0.260719
 17                 89    4.134831       0.359189
 27                 55    4.345455       0.408084
 10                117    4.401709       0.190182
 11                115    4.408696       0.435007
 3                 325    4.415385       0.440974
 5                 244    4.430328       0.462911
-1                1614    4.439901       0.413156
 21                 72    4.458333       0.646165
 18                 85    4.494118       0.473841
 4                 303    4.564356       0.322095


In [27]:
print("=== TOPIC 29 ===")
topic_29 = bertopic_sample_df[bertopic_sample_df["ml_topic"] == 29]
for text in topic_29["review_text"].head(15):
    print(f"- {text[:150]}")

print("\n=== TOPIC 20 ===")
topic_20 = bertopic_sample_df[bertopic_sample_df["ml_topic"] == 20]
for text in topic_20["review_text"].head(15):
    print(f"- {text[:150]}")

=== TOPIC 29 ===
- Väga hea äpp, töötab ka Graphene OS-iga.
- Stopped working on GrapheneOS. No longer usable. Cancelled my account.
- It's not working with GrapheneOS
- Unusable on GrapheneOS
- Can't use the app with GrapheneOS
- Why can't I use it on GrapheneOS?🙄
- Because of an incorrectly implemented device integrity check based on the anti-competitive Play Integrity API. REVOLUT APP HAS BANNED GRAPHENEOS USERS
- Blocking GrapheneOS was a mistake and they should drop the "pretend to be a secure bank" act in front of the wrong audience. I should have closed it l
- Broken on custom roms, including grapheneos that is more secure than any other mobile os
- Was using this for years as a metal customer. Suddenly they blocked my OS (grapheneos, probably the most secure OS out there) now, and I can't even ca
- Nonsense decision to forbid usage of graphene os, yet allow super outdated devices. Works nonsense is just too soft, decision is plain stupid. Whoever
- I use Revolut for more than 5

## BERTopic Analysis — Final Findings

**Method:** Unsupervised ML topic modeling (BERTopic) on a 20,000-review sample, used to 
discover topics directly from text without predefined categories — complementing the 
manual keyword-based theme analysis performed earlier.

### Reproducibility Note

Initial runs of BERTopic produced inconsistent topic counts across different samples (19 
vs. 58 topics), a known limitation of density-based clustering (UMAP + HDBSCAN) on datasets 
with a heavy positive skew (~88% positive reviews). Rather than rely on surface-level topic 
keywords — which were often generic and uninformative — clusters were instead ranked by 
**average star rating**, allowing genuine negative clusters to be identified directly from 
the data regardless of how their keyword summary looked.

### Key Finding: A Specific, Previously Unidentified Complaint — GrapheneOS Compatibility

| Metric | Value |
|---|---|
| Cluster size | 54 reviews |
| Average rating | 1.13 / 5 |
| Average sentiment score | -0.05 |

Manual inspection of this cluster's actual review text revealed a **specific, technical 
complaint not captured by any manual keyword theme**: users of **GrapheneOS** (a 
privacy/security-focused custom Android operating system) reported being unable to use the 
Revolut app, due to Revolut's Play Integrity API security checks blocking non-standard 
Android builds.

**Notable pattern in this cluster:** several reviewers explicitly identified themselves as 
long-tenured, premium ("Metal") customers (e.g., "using this for more than 5 years," "was 
using this for years as a metal customer") — suggesting this issue disproportionately 
affects a technically sophisticated, high-value, loyal customer segment rather than casual 
users.

### Business Implication

This finding was only surfaced through unsupervised topic modeling — it was not part of the 
original manually-defined keyword themes, since "GrapheneOS" or "Play Integrity API" were 
not terms anticipated in advance. This demonstrates the practical value of unsupervised ML 
as a complement to hypothesis-driven keyword analysis: it can surface specific, actionable 
issues that a predefined keyword list would miss entirely.

**Recommendation:** given the small but vocal, high-tenure, premium-tier nature of affected 
users, a targeted compatibility review (rather than dismissal) is warranted — losing 
security-conscious power users carries disproportionate reputational risk relative to the 
small review volume involved.

### Secondary Cluster (Topic 20)

A second negative cluster (76 reviews, avg rating 1.34) consisted almost entirely of short, 
generic negative exclamations ("Terrible," "Disaster," "Horrible") without specific 
complaint content — an artifact of very short reviews clustering by tone rather than topic, 
not an actionable finding on its own.

### Methodological Takeaway

Surface-level topic keywords proved unreliable for identifying genuinely negative clusters 
in this dataset — sorting clusters by actual average rating, then manually reviewing sample 
text, was necessary to separate meaningful findings (GrapheneOS) from clustering artifacts 
(generic positive/negative phrase groupings).

In [28]:
print("=== TOPIC 2 (400 reviews, avg rating 3.66) ===")
topic_2 = bertopic_sample_df[bertopic_sample_df["ml_topic"] == 2]
for text in topic_2["review_text"].head(20):
    print(f"- {text[:150]}")

=== TOPIC 2 (400 reviews, avg rating 3.66) ===
- Muy seguro y cómodo.
- 👍 Jest dobrze
- La app es eficiente, funciona muy bien. Hasta ahora no he tenido ningún inconveniente
- Super! Poate scapam de bănci pe viitor ;)
- gud
- Bun
- A++
- Goyyd fervice
- Greeeeaaat!!!!!
- Płacenie za granicą nigdy nie było tańsze i prostsze. Prosta wymiana, przelewy w jakiej chcesz walucie, prościej i taniej chyba się nie da.
- Ich benutze es als Sparkonto
- Excelent
- Bosh
- Brill can't fault
- Excelente
- Ļoti ērti lietojama aplikācija
- Ușoară și rapidă ☺️
- Update: od dłuższego czasu aplikacja działa bezbłędnie. Możliwości dokonywania chargeback'ów oraz blokowania subskrypcji nie raz uratowały mój portfel
- Hai ca e bun pt smecherie asa
- Tyu


In [29]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
bertopic_model_v2 = BERTopic(min_topic_size=20, umap_model=umap_model)  # lowered from 50 to 20

topics_v2, probs_v2 = bertopic_model_v2.fit_transform(docs)
bertopic_sample_df["ml_topic_v2"] = topics_v2

topic_sentiment_v2 = bertopic_sample_df.groupby("ml_topic_v2").agg(
    review_count=("rating", "count"),
    avg_rating=("rating", "mean"),
    avg_sentiment=("sentiment_score", "mean")
).sort_values("avg_rating")

print(topic_sentiment_v2.head(20))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2028.34it/s]


             review_count  avg_rating  avg_sentiment
ml_topic_v2                                         
126                    34    1.000000      -0.226471
75                     47    1.021277      -0.190181
74                     47    1.021277      -0.128398
156                    28    1.035714      -0.065046
32                     86    1.069767      -0.429023
18                    151    1.072848      -0.442796
14                    167    1.095808      -0.412184
139                    31    1.096774       0.089461
59                     54    1.129630      -0.052506
80                     46    1.130435       0.005052
120                    36    1.138889      -0.144136
85                     45    1.155556      -0.228478
104                    40    1.175000      -0.436057
89                     43    1.186047      -0.458219
93                     43    1.186047       0.020237
37                     76    1.197368      -0.084116
103                    40    1.225000      -0.

In [30]:
# Check the biggest negative clusters (most reviews = most common issues)
for topic_num in [18, 14, 16, 23, 32]:
    print(f"\n=== TOPIC {topic_num} ===")
    topic_reviews = bertopic_sample_df[bertopic_sample_df["ml_topic_v2"] == topic_num]
    print(f"Size: {len(topic_reviews)}, Avg rating: {topic_reviews['rating'].mean():.2f}")
    for text in topic_reviews["review_text"].head(10):
        print(f"- {text[:150]}")


=== TOPIC 18 ===
Size: 151, Avg rating: 1.07
- Legit
- Horrible scam! I tried 3 separate times to move the money to my bank account but was declined each time and to boot I only had the loan for less than 
- Some times they do cheating
- Scam
- Party to a scam which they have refused to assist in recovery.
- One of the worst banks I have come cross zero support for fraud avoid there Moto is contact the fraudsters they will give your money back absolute jok
- Scam! Signed up because it prompted you get 3 months of Tinder if you deposit £1. Deposited £1 and it's only offering Tinder if I go with the £55/mo o
- this is the worst bank i ever experienced. they just closed my account without giving me any reason and when i asked them they said unfortunately they
- Worst bank so far, just 1 month after using the bank, a message came saying they want to review the account and i can't use the acc for 14 days and i 
- Terrible firm. They won't let me close my account and they won't refund me my

## BERTopic Analysis — Final Findings

**Method:** Unsupervised ML topic modeling (BERTopic) on a 20,000-review sample. Rather than 
trust surface-level topic keywords (which proved unreliable — often generic across both 
positive and negative reviews), clusters were ranked by **average star rating**, then 
manually inspected to identify genuinely negative, specific complaint patterns. A second 
pass with a lower clustering threshold (`min_topic_size=20`, down from 50) was used to 
surface more granular clusters.

### Six Distinct Complaint Clusters Identified

| Topic | Size | Avg Rating | Complaint Category |
|---|---|---|---|
| 14 | 167 | 1.10 | Fraud / unauthorized transactions — victims report inadequate recovery support |
| 16 | 153 | 1.23 | Login & authentication failures — broken passcode reset, crashes on fingerprint login |
| 23 | 131 | 1.26 | Identity verification failures — no fallback method, unresolved for months |
| 18 | 151 | 1.07 | Scam allegations & failed fraud recovery — customers describe support as unhelpful |
| 32 | 86 | 1.07 | Unexplained account blocks/suspensions — including Premium/Metal-tier customers |
| 29 | 54 | 1.13 | GrapheneOS compatibility — Play Integrity API blocks privacy-focused custom Android OS |

### Why This Matters

These six clusters represent **distinct, specific operational failure modes** — not a single 
generic "poor support" issue, but genuinely different problems requiring different fixes.

### Cross-Validation

Topic 32 (unexplained account blocks) independently confirms the `account_freeze` finding 
from the manual keyword-based theme analysis — restoring BERTopic as a genuine third 
validation method for that finding.

### Business Recommendations

1. Fraud

## Reply Rate by Cluster

In [31]:
print("Reply rates for key negative clusters:")
for topic_num in [14, 16, 18, 23, 32, 29]:
    topic_data = bertopic_sample_df[bertopic_sample_df["ml_topic_v2"] == topic_num]
    reply_rate = topic_data["got_reply"].mean()
    print(f"Topic {topic_num}: {len(topic_data)} reviews, reply rate = {reply_rate:.1%}")

overall_reply_rate = bertopic_sample_df["got_reply"].mean()
print(f"\nOverall dataset reply rate (for comparison): {overall_reply_rate:.1%}")

Reply rates for key negative clusters:
Topic 14: 167 reviews, reply rate = 45.5%
Topic 16: 153 reviews, reply rate = 62.7%
Topic 18: 151 reviews, reply rate = 57.0%
Topic 23: 131 reviews, reply rate = 54.2%
Topic 32: 86 reviews, reply rate = 38.4%
Topic 29: 109 reviews, reply rate = 2.8%

Overall dataset reply rate (for comparison): 24.5%


## App Version Correlation

In [32]:
print("Top app versions per key negative cluster:\n")
for topic_num in [14, 16, 18, 23, 32]:
    print(f"=== Topic {topic_num} ===")
    topic_data = bertopic_sample_df[bertopic_sample_df["ml_topic_v2"] == topic_num]
    print(topic_data["reviewCreatedVersion"].value_counts().head(5))
    print()

Top app versions per key negative cluster:

=== Topic 14 ===
reviewCreatedVersion
Unknown    36
7.32        5
7.41.2      3
6.35        3
7.12.1      3
Name: count, dtype: int64

=== Topic 16 ===
reviewCreatedVersion
Unknown    32
10.37       4
10.5.2      3
9.17        3
7.11        3
Name: count, dtype: int64

=== Topic 18 ===
reviewCreatedVersion
Unknown    35
10.1        3
8.95        2
10.6        2
7.7.1       2
Name: count, dtype: int64

=== Topic 23 ===
reviewCreatedVersion
Unknown    30
10.106      3
10.11.1     2
5.54        2
6.1         2
Name: count, dtype: int64

=== Topic 32 ===
reviewCreatedVersion
Unknown     13
6.23         4
7.3          3
5.51         2
10.109.1     2
Name: count, dtype: int64



## What Drives 5-Star Loyalty (top of the sorted list, opposite end)

In [33]:
topic_sentiment_v2_sorted_desc = bertopic_sample_df.groupby("ml_topic_v2").agg(
    review_count=("rating", "count"),
    avg_rating=("rating", "mean")
).sort_values("avg_rating", ascending=False)

print("Top positive clusters (excluding -1 unclassified):")
print(topic_sentiment_v2_sorted_desc[topic_sentiment_v2_sorted_desc.index != -1].head(10))

Top positive clusters (excluding -1 unclassified):
             review_count  avg_rating
ml_topic_v2                          
149                    29    5.000000
131                    34    4.970588
140                    31    4.967742
152                    29    4.965517
154                    29    4.965517
41                     68    4.955882
101                    40    4.950000
112                    39    4.948718
134                    33    4.939394
70                     49    4.938776


In [35]:
# Replace X with the top positive topic number found above
positive_topic = bertopic_sample_df[bertopic_sample_df["ml_topic_v2"] == 149]
for text in positive_topic["review_text"].head(15):
    print(f"- {text[:150]}")

- The best
- The best
- the best
- The best one
- The best
- The best
- The best
- Probably the best
- the best
- The best
- the best
- The Best
- the best
- The best
- the best


## BERTopic Analysis — Complete Summary of Findings

**Method:** Unsupervised ML topic modeling (BERTopic) on a 20,000-review sample. Surface-level 
topic keywords proved unreliable (often generic across both positive and negative reviews), 
so clusters were instead ranked by average star rating and manually inspected. A lower 
clustering threshold (`min_topic_size=20`, down from 50) was used to surface more granular, 
specific complaint patterns.

### Six Distinct Complaint Clusters Identified

| Topic | Size | Avg Rating | Reply Rate | Complaint Category |
|---|---|---|---|---|
| 14 | 167 | 1.10 | 45.5% | Fraud / unauthorized transactions |
| 16 | 153 | 1.23 | 62.7% | Login & authentication failures |
| 23 | 131 | 1.26 | 54.2% | Identity verification failures |
| 18 | 151 | 1.07 | 57.0% | Scam allegations & failed fraud recovery |
| 32 | 86 | 1.07 | 38.4% | Unexplained account blocks/suspensions |
| 29 | 54-109 | 1.13 | **2.8%** | GrapheneOS compatibility |

*(Overall dataset reply rate, for comparison: 24.5%)*

### Finding 1: Distinct, Specific Complaint Categories
Six genuinely different operational issues, each requiring a different fix.

### Finding 2: Reply Rate Reveals a Prioritization Gap
GrapheneOS complaints receive a reply only 2.8% of the time — far below even the baseline 
rate (24.5%) — despite affecting long-tenured, premium customers.

### Finding 3: App Version — No Single Root Cause
Complaints spread across many versions — persistent, unresolved issues, not a one-time bug.

### Finding 4: "What Drives 5-Star Loyalty" — Null Result
Only generic repetitive phrases found; a clustering artifact, not a genuine insight.

### Cross-Validation
Topic 32 independently confirms the `account_freeze` theme from manual keyword analysis.

### Business Recommendations
1. Fraud response overhaul
2. Fix known technical bugs (passcode/fingerprint login)
3. Verification fallback path
4. Transparency in account suspensions
5. GrapheneOS support gap — most actionable finding
6. Root-cause investigation, not patching, given lack of version concentration

### Methodological Takeaways
- Surface-level keywords unreliable; sorting by rating was necessary
- Lower granularity revealed findings the default threshold missed
- Not every avenue yields a finding — null results reported honestly
- Reply rate cross-referenced against complaint category surfaced a hidden prioritization gap

## Transformer Sentiment Analysis

In [6]:
pip install transformers torch --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english", 
    truncation=True
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3779.80it/s]


## Run on a sample (transformers are much slower than VADER, so we sample first)

In [9]:
sample_df = df.dropna(subset=["review_text"]).sample(n=5000, random_state=42)
texts = sample_df["review_text"].astype(str).tolist()

results = sentiment_pipeline(texts, batch_size=32)
sample_df["transformer_label"] = [r["label"] for r in results]
sample_df["transformer_score"] = [r["score"] for r in results]

sample_df[["review_text", "rating", "sentiment_label", "transformer_label"]].head(10)

,review_text,rating,sentiment_label,transformer_label
23443,This Revolut App should not be allowed to prov...,1,neutral,NEGATIVE
226786,"Absolutely fantastic! Easy to use, friendly in...",5,positive,POSITIVE
182462,I'm new on this app and tell now it's working ...,5,positive,POSITIVE
78120,We went away best card I have had,5,positive,POSITIVE
61514,Perfect,5,positive,POSITIVE
168038,Support is quite bad. Long responses and cant ...,1,negative,NEGATIVE
121920,I'm happy and excited with the support. So qui...,5,positive,POSITIVE
201726,This is a total rip off scam service.. you don...,1,positive,NEGATIVE
29313,Revolut declined my credit card application wi...,1,negative,NEGATIVE
275495,Best app I've used in ages. Downside is revolu...,4,positive,POSITIVE


## Compare VADER vs Transformer agreement

In [10]:
sample_df["vader_vs_transformer_match"] = (
    (sample_df["sentiment_label"] == "positive") & (sample_df["transformer_label"] == "POSITIVE")
) | (
    (sample_df["sentiment_label"] == "negative") & (sample_df["transformer_label"] == "NEGATIVE")
)
print(f"Agreement rate: {sample_df['vader_vs_transformer_match'].mean():.1%}")

Agreement rate: 79.4%


## Save

In [11]:
sample_df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_transformer_sentiment.csv", index=False)
print("Saved.")

Saved.


## VADER vs. Transformer Sentiment — Comparison Findings

**Method:** Ran a pre-trained transformer model (DistilBERT, fine-tuned for sentiment 
classification) on a 5,000-review sample, and compared its output against the VADER 
sentiment labels generated earlier in this project.

| Metric | Value |
|---|---|
| Reviews compared | 5,000 (sample) |
| Agreement rate | 79.4% |
| Disagreement rate | 20.6% |

**What the models are:**
- **VADER** — a rule-based tool that scores sentiment by matching words against a fixed 
  dictionary of positive/negative terms. Fast, but has no real understanding of context.
- **Transformer (DistilBERT)** — a deep learning model trained on millions of examples, 
  able to interpret full sentence context, negation, and subtler phrasing.

### Key Finding

Manual inspection of disagreements showed a clear pattern: **the transformer model was 
more accurate specifically on negative (1-star) reviews.** Examples:

| Review (excerpt) | Rating | VADER | Transformer |
|---|---|---|---|
| "This Revolut App should not be allowed to prov..." | 1★ | Neutral | Negative (correct) |
| "This is a total rip off scam service.." | 1★ | Positive | Negative (correct) |

In both cases, VADER misclassified a clearly negative complaint, while the transformer 
correctly identified it as negative.

### Why this matters

This suggests the original VADER-based sentiment analysis may have **understated the true 
volume of negative sentiment**, particularly for reviews using indirect or non-dictionary 
negative language (e.g., "scam," "rip off"). Since these misclassifications specifically 
affected 1-star reviews — the most business-critical category for this analysis — the 
transformer-based labels should be treated as the more reliable source going forward.

### Decision

Going forward, transformer-based sentiment labels will be used as the primary sentiment 
measure for dashboard visuals and business recommendations, with VADER results retained 
only as a secondary reference point.

**Limitation:** this comparison was run on a 5,000-review sample, not the full 112,000+ 
dataset, due to the transformer model's higher processing time per review.

## Run it in longer dataset

In [13]:
sample_df_large = df.dropna(subset=["review_text"]).sample(n=30000, random_state=42)
texts = sample_df_large["review_text"].astype(str).tolist()

results = sentiment_pipeline(texts, batch_size=32)
sample_df_large["transformer_label"] = [r["label"] for r in results]
sample_df_large["transformer_score"] = [r["score"] for r in results]

sample_df_large.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_transformer_sentiment_large.csv", index=False)
print(f"Saved {len(sample_df_large)} reviews with transformer sentiment.")

Saved 30000 reviews with transformer sentiment.


In [14]:
print("Transformer sentiment distribution:")
print(sample_df_large["transformer_label"].value_counts())
print(f"\nPercentage negative: {(sample_df_large['transformer_label'] == 'NEGATIVE').mean():.1%}")

print("\nVADER sentiment distribution (same reviews):")
print(sample_df_large["sentiment_label"].value_counts())

Transformer sentiment distribution:
transformer_label
POSITIVE    23027
NEGATIVE     6973
Name: count, dtype: int64

Percentage negative: 23.2%

VADER sentiment distribution (same reviews):
sentiment_label
positive    23441
neutral      3463
negative     3096
Name: count, dtype: int64


In [15]:
one_star = sample_df_large[sample_df_large["rating"] == 1]

print(f"Among {len(one_star)} one-star reviews:")
print(f"VADER correctly identified as negative: {(one_star['sentiment_label'] == 'negative').mean():.1%}")
print(f"Transformer correctly identified as negative: {(one_star['transformer_label'] == 'NEGATIVE').mean():.1%}")

Among 3680 one-star reviews:
VADER correctly identified as negative: 57.2%
Transformer correctly identified as negative: 95.9%


## Ground-Truth Accuracy Check: VADER vs. Transformer

**Method:** Using star rating as ground truth (a 1-star review is, by definition, a negative 
experience), tested how often each sentiment method correctly classified confirmed 1-star 
reviews as "negative."

| Metric | Value |
|---|---|
| One-star reviews tested | 3,680 |
| VADER correctly identified as negative | 57.2% |
| Transformer correctly identified as negative | 95.9% |
| **Improvement** | **+38.7 percentage points** |

### Interpretation

VADER, a lexicon/rule-based sentiment tool, **missed 42.8% of confirmed negative reviews** — 
misclassifying them as neutral or even positive, likely because these reviews used indirect 
or context-dependent negative language (e.g., "scam," "rip off," sarcasm) not captured by 
VADER's fixed word dictionary.

The transformer model (DistilBERT, fine-tuned for sentiment) correctly identified 95.9% of 
the same reviews, demonstrating far greater reliability when validated against real, 
unambiguous ground truth.

### Why this matters for the project

This is a stronger and more defensible test than simple agreement rate between the two 
models, because it uses an independent, verifiable source of truth (the star rating itself) 
rather than comparing two uncertain methods against each other.

**Conclusion:** the original VADER-based sentiment analysis likely understated the true 
volume of negative sentiment in the dataset. The transformer model is adopted as the primary 
sentiment classification method going forward; VADER results are retained only as an initial, 
lower-fidelity baseline.

**Methodological takeaway:** this exercise demonstrates the value of benchmarking a fast, 
simple NLP method against a more sophisticated model using ground truth, rather than 
assuming the simpler method is adequate without testing it.

## Finding the GrapheneOS Complaint cluser and timeline

In [36]:
graphene_topic = bertopic_sample_df[bertopic_sample_df["ml_topic_v2"] == 29]
print(f"Total GrapheneOS complaints: {len(graphene_topic)}")
print(f"\nApp version distribution:")
print(graphene_topic["reviewCreatedVersion"].value_counts().head(15))
print(f"\nDate range of these complaints:")
print(f"Earliest: {graphene_topic['review_date'].min()}")
print(f"Latest: {graphene_topic['review_date'].max()}")

Total GrapheneOS complaints: 109

App version distribution:
reviewCreatedVersion
Unknown     21
10.115.2     2
8.35         2
8.0.1        2
8.64.1       2
6.23         2
10.123       2
8.96.2       2
8.5          2
8.42.1       2
10.105       1
7.23.3       1
6.4          1
8.54         1
8.52         1
Name: count, dtype: int64

Date range of these complaints:
Earliest: 1/10/2020 21:28
Latest: 9/8/2023 9:11


In [37]:
graphene_topic_sorted = bertopic_sample_df[bertopic_sample_df["ml_topic_v2"] == 29].sort_values("review_date", ascending=False)
print(graphene_topic_sorted[["review_date", "review_text"]].head(15).to_string())

            review_date review_text
98433     9/8/2023 9:11        Good
197314   9/7/2020 22:06        Good
127765   9/25/2022 8:11        Good
96838   9/22/2023 12:17        Good
128090  9/21/2022 13:11        Good
97105   9/20/2023 10:32        Good
160068  9/20/2021 15:45        Good
160707   9/14/2021 7:43        Good
266697   9/1/2018 11:43        Good
101232   8/8/2023 22:42        Good
231859  8/31/2019 16:19        Good
231887  8/31/2019 12:35        Good
200751    8/3/2020 3:54        Good
99594   8/27/2023 13:59        Good
25269   8/22/2025 16:29        good


In [38]:
topic_29_all = bertopic_sample_df[bertopic_sample_df["ml_topic_v2"] == 29]

# Check how many actually mention grapheneos-related terms vs generic
graphene_related = topic_29_all[topic_29_all["review_text"].str.contains("graphene|GrapheneOS|integrity", case=False, na=False)]
print(f"Total in Topic 29: {len(topic_29_all)}")
print(f"Actually mention GrapheneOS/integrity: {len(graphene_related)}")
print(f"\nRating distribution of the GENUINE GrapheneOS subset:")
print(graphene_related["rating"].value_counts())
print(f"\nDate range of GENUINE GrapheneOS mentions:")
print(f"{graphene_related['review_date'].min()} to {graphene_related['review_date'].max()}")

Total in Topic 29: 109
Actually mention GrapheneOS/integrity: 0

Rating distribution of the GENUINE GrapheneOS subset:
Series([], Name: count, dtype: int64)

Date range of GENUINE GrapheneOS mentions:
nan to nan


In [39]:
topic_29_correct = bertopic_sample_df[bertopic_sample_df["ml_topic"] == 29]  # NOTE: ml_topic, not ml_topic_v2

graphene_related = topic_29_correct[topic_29_correct["review_text"].str.contains("graphene|integrity", case=False, na=False)]
print(f"Total in Topic 29 (ml_topic): {len(topic_29_correct)}")
print(f"Actually mention GrapheneOS/integrity: {len(graphene_related)}")
print(f"\nRating distribution:")
print(graphene_related["rating"].value_counts())
print(f"\nDate range:")
print(f"{graphene_related['review_date'].min()} to {graphene_related['review_date'].max()}")

Total in Topic 29 (ml_topic): 54
Actually mention GrapheneOS/integrity: 52

Rating distribution:
rating
1    49
5     1
3     1
2     1
Name: count, dtype: int64

Date range:
1/11/2025 21:52 to 9/30/2025 22:54


In [41]:
graphene_final = topic_29_correct[topic_29_correct["review_text"].str.contains("graphene|integrity", case=False, na=False)].copy()
graphene_final["review_date"] = pd.to_datetime(graphene_final["review_date"])
graphene_final["month_year"] = graphene_final["review_date"].dt.to_period("M").astype(str)

monthly_complaints = graphene_final.groupby("month_year").size().reset_index(name="complaint_count")
monthly_complaints.to_csv("E:/project/Revolut-analysis-deashboard/Data/graphene_os_complaints_timeline.csv", index=False)
print(monthly_complaints)

   month_year  complaint_count
0     2024-10                1
1     2024-12               29
2     2025-01               11
3     2025-02                1
4     2025-04                2
5     2025-07                1
6     2025-08                1
7     2025-09                2
8     2025-10                1
9     2026-01                1
10    2026-02                1
11    2026-03                1


In [42]:
graphene_final = topic_29_correct[topic_29_correct["review_text"].str.contains("graphene|integrity", case=False, na=False)].copy()

print(f"Total complaints: {len(graphene_final)}")
print(f"Unique userNames: {graphene_final['userName'].nunique()}")
print(f"\nAny repeat users?")
print(graphene_final['userName'].value_counts().head(10))

Total complaints: 52
Unique userNames: 52

Any repeat users?
userName
Roland Helerand         1
HappyLeptiC             1
Iván Córdoba Donet      1
Miljana Dimitrijević    1
Floris Apon             1
Oliver                  1
Vitalie                 1
Octavian David          1
Joseph Kiely (Jay)      1
Ky                      1
Name: count, dtype: int64


In [43]:
print(f"App version distribution for confirmed GrapheneOS complaints:")
print(graphene_final["reviewCreatedVersion"].value_counts())

print(f"\nApp version distribution (appVersion field, if different):")
print(graphene_final["appVersion"].value_counts())

App version distribution for confirmed GrapheneOS complaints:
reviewCreatedVersion
10.56.2     15
Unknown      7
10.62.1      7
10.58.3      4
10.61        2
10.6         2
10.93        2
10.60.1      2
10.40.3      1
8.47         1
10.33.2      1
10.53.1      1
10.43.2      1
10.58.2      1
10.112.1     1
10.98        1
10.75        1
4.22.0       1
10.122.1     1
Name: count, dtype: int64

App version distribution (appVersion field, if different):
appVersion
10.56.2     15
Unknown      7
10.62.1      7
10.58.3      4
10.61        2
10.6         2
10.93        2
10.60.1      2
10.40.3      1
8.47         1
10.33.2      1
10.53.1      1
10.43.2      1
10.58.2      1
10.112.1     1
10.98        1
10.75        1
4.22.0       1
10.122.1     1
Name: count, dtype: int64


In [44]:
version_counts = graphene_final["reviewCreatedVersion"].value_counts().reset_index()
version_counts.columns = ["version", "complaint_count"]
version_counts.to_csv("E:/project/Revolut-analysis-deashboard/Data/graphene_os_versions.csv", index=False)
print(version_counts)

     version  complaint_count
0    10.56.2               15
1    Unknown                7
2    10.62.1                7
3    10.58.3                4
4      10.61                2
5       10.6                2
6      10.93                2
7    10.60.1                2
8    10.40.3                1
9       8.47                1
10   10.33.2                1
11   10.53.1                1
12   10.43.2                1
13   10.58.2                1
14  10.112.1                1
15     10.98                1
16     10.75                1
17    4.22.0                1
18  10.122.1                1


In [45]:
graphene_rating_dist = graphene_final["rating"].value_counts().sort_index().reset_index()
graphene_rating_dist.columns = ["rating", "count"]
graphene_rating_dist.to_csv("E:/project/Revolut-analysis-deashboard/Data/graphene_os_rating_distribution.csv", index=False)
print(graphene_rating_dist)

   rating  count
0       1     49
1       2      1
2       3      1
3       5      1
